In [1]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
import matplotlib as mpl

For now using old regridded maps, but will need to redo that for CHIME once I have the new files from cedar (hpx dec calcs instead of plate caree)

## Make spatially filtered versions of the maps and save FITS cubes

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'data/'

Rlims = [1,2,100,105]
filt_version = 'Sep26_highpass_mask'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes+'.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_'+stokes+'.fits'

    print(chime_file)
    print(dragons_file)
    dragons_chime_conv_combine.band_pass(chime_file, dragons_file, dir_out+sub_dir, stokes, Rlims, filt_version)

## Calculate the Jy/beam to K conversion factors for CHIME using spatially filtered data

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_filtered_highpass_Sep26/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_filtered_'+stokes+'_Sep26_highpass_mask.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_filtered_'+stokes+'_Sep26_highpass_mask.fits'

    dragons_chime_conv_combine.compare_chime_dragons(chime_file, dragons_file, dir_out+sub_dir,stokes)

# Organize the TT plots:

freqs_new = np.linspace(512.5,729.5,218)
for freq in freqs_new[0:218]:
    !{'mkdir '+dir_out+sub_dir+str(int(np.floor(freq)))}
    !{'mv '+dir_out+sub_dir+'*'+str(freq)+'*.png '+dir_out+sub_dir+str(int(np.floor(freq)))+'/'}

In [ ]:
def conversion_vs_freq(idx,paramsQ,paramsU):

    fs = 14
    
    print(paramsQ['decs'][idx])

    fig,ax = plt.subplots(2,1,figsize=(16,8))

    ax[0].scatter(paramsQ['freqs'],paramsQ['slopes'][:,idx],label='Stokes Q',s=20)
    ax[0].scatter(paramsU['freqs'],paramsU['slopes'][:,idx],label='Stokes U',s=20)
    ax[0].set_ylim(0,2)
    ax[0].set_ylabel('Slope [K/Jy/beam]',fontsize=fs)

    ax[1].scatter(paramsQ['freqs'],paramsQ['ints'][:,idx],label='Stokes Q',s=20)
    ax[1].scatter(paramsU['freqs'],paramsU['ints'][:,idx],label='Stokes U',s=20)
    ax[1].set_ylim(-0.5,0.5)
    ax[1].set_ylabel('Intercept [K]',fontsize=fs)

    for i in range(0,2):
        ax[i].set_xlim(510,730)
        ax[i].grid()
        ax[i].set_xlabel('Frequency',fontsize=fs)
        ax[i].legend(fontsize=fs)
        ax[i].tick_params(axis='both', labelsize=fs)

    dec_string = str(paramsQ['decs'][idx])
    plt.suptitle('DRAGONS/CHIME conversion factor, dec='+dec_string,fontsize=fs+4)
    
    return


def conversion_vs_dec(idx,paramsQ,paramsU):

    fs = 14
    
    print(paramsQ['freqs'][idx])

    fig,ax = plt.subplots(2,1,figsize=(16,8))

    ax[0].scatter(paramsQ['decs'],paramsQ['slopes'][idx,:],label='Stokes Q',s=10)
    ax[0].scatter(paramsU['decs'],paramsU['slopes'][idx,:],label='Stokes U',s=10)
    ax[0].set_ylim(0,2)
    ax[0].set_ylabel('Slope [K/Jy/beam]',fontsize=fs)

    ax[1].scatter(paramsQ['decs'],paramsQ['ints'][idx,:],label='Stokes Q',s=10)
    ax[1].scatter(paramsU['decs'],paramsU['ints'][idx,:],label='Stokes U',s=10)
    ax[1].set_ylim(-0.5,0.5)
    ax[1].set_ylabel('Intercept [K]',fontsize=fs)

    for i in range(0,2):
        ax[i].set_xlim(10,90)
        ax[i].grid()
        ax[i].set_xlabel('Declination',fontsize=fs)
        ax[i].legend(fontsize=fs)
        ax[i].tick_params(axis='both', labelsize=fs)

    freq_string = str(paramsQ['freqs'][idx])
    plt.suptitle('DRAGONS/CHIME conversion factor, '+freq_string+' MHz',fontsize=fs+4)
        
    return 

In [ ]:
def conversion_vs_dec_smooth(idx,paramsQ,paramsU,paramsQ_smooth,paramsU_smooth):

    fs = 14
    
    print(paramsQ['freqs'][idx])

    fig,ax = plt.subplots(2,1,figsize=(16,8))

    ax[0].scatter(paramsQ['decs'],paramsQ['slopes'][idx,:],label='Stokes Q',s=10)
    ax[0].scatter(paramsU['decs'],paramsU['slopes'][idx,:],label='Stokes U',s=10)
    ax[0].scatter(paramsQ_smooth['decs'],paramsQ_smooth['slopes'][idx,:],s=2,color='blue')
    ax[0].scatter(paramsU_smooth['decs'],paramsU_smooth['slopes'][idx,:],s=2,color='red')
    ax[0].set_ylim(0,2)
    ax[0].set_ylabel('Slope [K/Jy/beam]',fontsize=fs)

    ax[1].scatter(paramsQ['decs'],paramsQ['ints'][idx,:],label='Stokes Q',s=10)
    ax[1].scatter(paramsU['decs'],paramsU['ints'][idx,:],label='Stokes U',s=10)
    ax[1].scatter(paramsQ_smooth['decs'],paramsQ_smooth['ints'][idx,:],s=2,color='blue')
    ax[1].scatter(paramsU_smooth['decs'],paramsU_smooth['ints'][idx,:],s=2,color='red')
    ax[1].set_ylim(-0.5,0.5)
    ax[1].set_ylabel('Intercept [K]',fontsize=fs)

    for i in range(0,2):
        ax[i].set_xlim(10,90)
        ax[i].grid()
        ax[i].set_xlabel('Declination',fontsize=fs)
        ax[i].legend(fontsize=fs)
        ax[i].tick_params(axis='both', labelsize=fs)

    freq_string = str(paramsQ['freqs'][idx])
    plt.suptitle('DRAGONS/CHIME conversion factor, '+freq_string+' MHz',fontsize=fs+4)
        
    return 

## Make CHIME maps in K

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_filtered_highpass_Sep26/'

for stokes in ['Q','U']:

    chime_file   = '/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_'+stokes+'.fits'
    conversion_file = '/srv/data/gmims/gmims-lbn/experiments/'+sub_dir+'/chime_vs_dragons_stokes'+stokes+'.npz'

    dragons_chime_conv_combine.chimeK(chime_file, conversion_file, dir_out+sub_dir, stokes)

## Plot conversion factors

In [ ]:
paramsQ = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_vs_dragons_stokesQ.npz')
paramsU = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_vs_dragons_stokesU.npz')
print(paramsQ.keys())
print(paramsU.keys())

print(paramsQ['decs'].shape)
print(paramsQ['slopes'].shape)
print(paramsQ['freqs'].shape)
print(paramsQ['ints'].shape)

print('')

paramsQ_smooth = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_vs_dragons_stokesQ_smooth.npz')
paramsU_smooth = np.load('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_vs_dragons_stokesU_smooth.npz')
print(paramsQ_smooth.keys())
print(paramsU_smooth.keys())

print(paramsQ_smooth['decs'].shape)
print(paramsQ_smooth['slopes'].shape)
print(paramsQ_smooth['freqs'].shape)
print(paramsQ_smooth['ints'].shape)

In [ ]:
for idx in range(0,218):

    try:
        #conversion_vs_dec(idx,paramsQ,paramsU)
        #conversion_vs_dec(idx,paramsQ_smooth,paramsU_smooth)
        conversion_vs_dec_smooth(idx,paramsQ,paramsU,paramsQ_smooth,paramsU_smooth)
        freq_string = str(paramsQ['freqs'][idx])
        plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/conversion_plots/conversion_vs_dec_'+freq_string+'.png',
                    facecolor='white')
    except:
        pass

In [ ]:
paramsQ['decs'][50]

In [ ]:
#idx_min = 801
#idx_max = 1412
#print(paramsQ['decs'][idx_min])
#print(paramsQ['decs'][idx_max])

#for idx in range(idx_min,idx_max):

for idx in [50]:
    try:
        conversion_vs_freq(idx,paramsQ,paramsU)
        dec_string = str(paramsQ['decs'][idx])
        plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/conversion_plots/conversion_vs_freq_'+dec_string+'.png')
    except:
        pass

## Combine CHIME and DRAGONS

In [ ]:
%%time

importlib.reload(dragons_chime_conv_combine)

dir_out = '/srv/data/gmims/gmims-lbn/experiments/'
sub_dir = 'results_filtered_highpass_Sep26/'

Rlims = [3,5]
filt_version = 'Sep26_highpass_mask'

for stokes in ['Q','U']:

    chime_file   = dir_out+sub_dir+'chime_K_'+stokes+'.fits'
    dragons_file = '/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_'+stokes+'.fits'
    
    print(chime_file)
    print(dragons_file)
    dragons_chime_conv_combine.combine(chime_file, dragons_file, dir_out+sub_dir, stokes, Rlims, filt_version)

## NaN out bad channels in all data sets

In [ ]:
idx_bad = np.concatenate([list(np.arange(2,8,1)),
                          [9,10,15],
                          list(np.arange(17,25,1)),
                          [27],
                          list(np.arange(30,78,1)),
                          [79],
                          list(np.arange(82,95,1)),
                          list(np.arange(105,130,1)),
                          [142,143],
                          list(np.arange(151,158,1)),
                          [159,160,163],
                          list(np.arange(180,186,1)),
                          list(np.arange(187,203,1)),
                          list(np.arange(206,216,1))
                         ])
print(idx_bad)

In [ ]:
stokes_list = ['Q','U']

out_dir = '/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/'

for stokes in stokes_list:

    files  = ['/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_K_'+stokes+'.fits',
              '/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_'+stokes+'.fits',
              '/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_dragons_'+stokes+'_Sep26_highpass_mask.fits']
    outname    = [out_dir+'chime/'+stokes+'_chime.fits',out_dir+'dragons/'+stokes+'_dragons.fits',out_dir+'dragons_chime/'+stokes+'_dragons_chime.fits']

    for i in range(0,3):

        print(files[i])
        print(outname[i])
        print('')
        
        hdu = fits.open(files[i])
        hdu[0].data[idx_bad] = np.nan

        fits.writeto(outname[i], hdu[0].data.astype('float32'), hdu[0].header, overwrite=True)
        

## Plot frequency cube

In [ ]:
hduQ = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/Q_dragons_chime.fits')
hduU = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/U_dragons_chime.fits')

Q = hduQ[0].data
U = hduU[0].data

wcs = WCS(hduQ[0].header).dropaxis(2)

freq = WCS(hduQ[0].header).all_pix2world(0,0,range(hduQ[0].data.shape[0]),0)[2]/1e6

#print(freq)

In [ ]:
crange = SkyCoord([24,0], [-20,90], frame=ICRS, unit=(u.hourangle, u.deg))
    
fs = 14


for j in range(0,218):
    fig = plt.figure(figsize=(20,13))
    ax1 = fig.add_subplot(211, projection=wcs.celestial)
    ax2 = fig.add_subplot(212, projection=wcs.celestial)
    
    im1 = ax1.imshow(Q[j],origin='lower',cmap='RdBu_r',vmin=-2,vmax=2)
    im2 = ax2.imshow(U[j],origin='lower',cmap='RdBu_r',vmin=-2,vmax=2)
    
    axs = [ax1,ax2]
    ims = [im1,im2]
    
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.8, pad=0.01)
        cbar.set_label('K', fontsize=fs)
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('dec',fontsize=fs)
    ax2.set_xlabel('RA',fontsize=fs)
    
    ax1.set_title('Stokes Q, '+str(freq[j])+' MHz')
    ax2.set_title('Stokes U, '+str(freq[j])+' MHz')
    
    #plt.tight_layout()
    
    plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/maps_'+str(freq[j])+'.png')

In [ ]:
def plot_2panel_maps(map1,map2,wcs,labels,title):

    fig = plt.figure(figsize=(18,8))
    fs = 16

    plt.subplots_adjust(hspace=0.0,left=0.05,right=0.999,top=0.95,bottom=0.05,wspace=0)

    crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))

    ax1 = fig.add_subplot(211, projection=wcs.celestial)
    ax2 = fig.add_subplot(212, projection=wcs.celestial)

    im1 = ax1.imshow(map1,vmin=-1,vmax=1,cmap='RdBu_r')
    im2 = ax2.imshow(map2,vmin=-1,vmax=1,cmap='RdBu_r')

    labels = [r'Stokes Q (Jy beam$^{-1}$)',r'Stokes U (Jy beam$^{-1}$)']

    axs = [ax1,ax2]
    ims = [im1,im2]

    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.82, pad=0.005, aspect=12)
        cbar.set_label(label=labels[i],fontsize=fs)
        cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('Declination',fontsize=fs)
        axs[i].tick_params(axis='both',labelsize=fs)
    
    ax2.set_xlabel('Right Ascension',fontsize=fs)

    ax1.set_title('Stacked maps',fontsize=fs+2)

    return

## Do RM synthesis

In [ ]:
np.savetxt('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt',freq*1e6)

In [ ]:
!rmsynth3d \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/Q_dragons_chime.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/U_dragons_chime.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt -v -d 0.5 -l 150 -t

In [ ]:
!rmsynth3d \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/Q_dragons.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/U_dragons.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt -v -d 0.5 -l 150 -t

In [ ]:
!rmsynth3d \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/Q_chime.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/U_chime.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt -v -d 0.5 -l 150 -t

## RM CLEAN

In [ ]:
!rmclean3d /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/FDF_tot_dirty.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/RMSF_tot.fits -v -c 0.6


In [ ]:
!rmclean3d /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/FDF_tot_dirty.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/RMSF_tot.fits -v -c 0.3


In [ ]:
!rmclean3d /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/FDF_tot_dirty.fits \
     /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/RMSF_tot.fits -v -c 0.6

In [ ]:
!rmtools_peakfitcube \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/FDF_clean_tot.fits \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/clean_ -v


In [ ]:
!rmtools_peakfitcube \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/FDF_clean_tot.fits \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/chime/clean_ -v


In [ ]:
!rmtools_peakfitcube \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/FDF_clean_tot.fits \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/frequencies.txt \
 /srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons/clean_ -v